# Projet 05 — Urbanisme et Environnement : Tendances des Permis a Bruxelles

**Question politique** : Comment evoluent les permis d'urbanisme et d'environnement a Bruxelles ? Y a-t-il une pression croissante sur l'environnement due au developpement immobilier, ou la ville regule-t-elle efficacement ses activites a impact environnemental ?

**Hypothese** : L'augmentation des permis d'urbanisme non accompagnee d'un renforcement des permis d'environnement signalerait une insuffisance de la protection environnementale face a la pression de construction.

---

**Auteur** : Kuate Joel Parfait  
**LinkedIn** : [joelparfaitkuate](https://be.linkedin.com/in/joelparfaitkuate/)  
**Contact** : hello@dhcompany.pro | 0465 55 71 09  
**Source de donnees** : Open Data Bruxelles — opendata.brussels.be  
**Datasets** :  
- `permis-environnement-delivres-vbx`  
- `permis-urbanisme-environnement-vbx`  
**Derniere mise a jour** : Mars 2026

---

> Kuate Joel Parfait — hello@dhcompany.pro | 0465 55 71 09  
> [linkedin.com/in/joelparfaitkuate](https://be.linkedin.com/in/joelparfaitkuate/)  
> www.axiatechnologie.com

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (13, 6)
plt.rcParams['font.family'] = 'DejaVu Sans'
print("Librairies chargees.")

## 1. Chargement et exploration des donnees

In [ ]:
# Permis d'environnement delivres
df_env = pd.read_csv('permis_environnement_delivres.csv', sep=';', encoding='utf-8-sig')
print("Permis environnement — shape:", df_env.shape)
print("Colonnes:", df_env.columns.tolist())
df_env.head(10)

In [ ]:
# Permis d'urbanisme et environnement
df_urba = pd.read_csv('permis_urbanisme_environnement.csv', sep=';', encoding='utf-8-sig')
print("Permis urbanisme — shape:", df_urba.shape)
print("Colonnes:", df_urba.columns.tolist())
df_urba.head(10)

In [ ]:
# Nettoyage
for df in [df_env, df_urba]:
    for col in df.columns:
        if any(x in col for x in ['Total', 'Année', 'Annee', 'annee']):
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
            except:
                pass

print("=== Permis environnement — annees ===")
annee_col_env = [c for c in df_env.columns if 'Ann' in c or 'ann' in c][0]
print(sorted(df_env[annee_col_env].dropna().unique().astype(int)))

print("\n=== Permis urbanisme — annees ===")
annee_col_urba = [c for c in df_urba.columns if 'Ann' in c or 'ann' in c][0]
print(sorted(df_urba[annee_col_urba].dropna().unique().astype(int)))

## 2. Evolution temporelle des permis d'environnement

In [ ]:
# Permis environnement : evolution par annee et domaine
domaine_col = [c for c in df_env.columns if 'Domaine' in c or 'domaine' in c][0]
total_col = [c for c in df_env.columns if 'Total global' in c or 'total' in c.lower()][0]

df_env_year = df_env.groupby([annee_col_env, domaine_col])[total_col].sum().reset_index()
df_env_pivot = df_env_year.pivot_table(index=annee_col_env, columns=domaine_col, values=total_col, aggfunc='sum').fillna(0)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Graph 1 : Evolution par domaine
df_env_pivot.plot(ax=axes[0], linewidth=2.5, marker='o', markersize=4)
axes[0].set_title("Evolution des permis d'environnement par domaine\n(Ville de Bruxelles)", 
                  fontweight='bold', fontsize=12)
axes[0].set_xlabel("Annee")
axes[0].set_ylabel("Nombre de permis delivres")
axes[0].legend(title="Domaine", bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
axes[0].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

# Graph 2 : Total par annee
total_env = df_env.groupby(annee_col_env)[total_col].sum().reset_index()
axes[1].fill_between(total_env[annee_col_env], total_env[total_col], alpha=0.7, color='steelblue')
axes[1].plot(total_env[annee_col_env], total_env[total_col], color='steelblue', linewidth=2, marker='o')
axes[1].set_title("Total annuel des permis d'environnement delivres\n(Ville de Bruxelles)", 
                  fontweight='bold', fontsize=12)
axes[1].set_xlabel("Annee")
axes[1].set_ylabel("Total permis delivres")
axes[1].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig('01_evolution_permis_env.png', dpi=150, bbox_inches='tight')
plt.show()
print("Graphique sauvegarde.")

## 3. Analyse des permis d'urbanisme

In [ ]:
# Categories principales des permis d'urbanisme
domaine_urba = [c for c in df_urba.columns if 'Domaine' in c][0]
cat_col_urba = [c for c in df_urba.columns if 'Cat' in c and 'gorie' in c and 'Code' not in c][0] if any('Cat' in c and 'gorie' in c and 'Code' not in c for c in df_urba.columns) else None
total_urba = [c for c in df_urba.columns if 'Total global' in c][0]

print("Domaines urbanisme:")
print(df_urba[domaine_urba].value_counts())
if cat_col_urba:
    print("\nTop categories:")
    print(df_urba.groupby(cat_col_urba)[total_urba].sum().sort_values(ascending=False).head(10))

In [ ]:
# Evolution permis urbanisme
df_urba_year = df_urba.groupby(annee_col_urba)[total_urba].sum().reset_index()

# Top categories urbanisme
if cat_col_urba:
    df_cat_urba = df_urba.groupby([annee_col_urba, cat_col_urba])[total_urba].sum().reset_index()
    top_cats = df_urba.groupby(cat_col_urba)[total_urba].sum().nlargest(5).index
    df_cat_top = df_cat_urba[df_cat_urba[cat_col_urba].isin(top_cats)]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Graph 1 : Total permis urbanisme par annee
axes[0].fill_between(df_urba_year[annee_col_urba], df_urba_year[total_urba], alpha=0.7, color='coral')
axes[0].plot(df_urba_year[annee_col_urba], df_urba_year[total_urba], color='coral', linewidth=2, marker='o')
axes[0].set_title("Evolution des permis d'urbanisme\n(Ville de Bruxelles)", fontweight='bold', fontsize=12)
axes[0].set_xlabel("Annee")
axes[0].set_ylabel("Total permis")
axes[0].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

# Graph 2 : Top categories
if cat_col_urba and len(df_cat_top) > 0:
    pivot_top = df_cat_top.pivot_table(index=annee_col_urba, columns=cat_col_urba, values=total_urba, aggfunc='sum').fillna(0)
    pivot_top.plot(ax=axes[1], linewidth=2.5, marker='o', markersize=4)
    axes[1].set_title("Top 5 categories de permis d'urbanisme\n(Ville de Bruxelles)", fontweight='bold', fontsize=12)
    axes[1].set_xlabel("Annee")
    axes[1].set_ylabel("Nombre de permis")
    axes[1].legend(title="Categorie", bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=7)
    axes[1].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
else:
    df_urba.groupby(domaine_urba)[total_urba].sum().sort_values().plot(kind='barh', ax=axes[1], color='coral')
    axes[1].set_title("Permis d'urbanisme par domaine\n(cumul)", fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig('02_evolution_permis_urba.png', dpi=150, bbox_inches='tight')
plt.show()
print("Graphique sauvegarde.")

## 4. Croisement : Comparaison permis urbanisme vs permis environnement

In [ ]:
# Croisement temporal : urbanisme vs environnement
# Fusionner les totaux annuels
df_comp_env = df_env.groupby(annee_col_env)[total_col].sum().reset_index()
df_comp_env.columns = ['Annee', 'Permis_Environnement']

df_comp_urba = df_urba.groupby(annee_col_urba)[total_urba].sum().reset_index()
df_comp_urba.columns = ['Annee', 'Permis_Urbanisme']

df_compare = pd.merge(df_comp_urba, df_comp_env, on='Annee', how='outer').fillna(0)
df_compare = df_compare.sort_values('Annee')
df_compare = df_compare[df_compare['Annee'] >= 2000]

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Graph 1 : Superposition volumes
ax1 = axes[0]
ax2 = ax1.twinx()
ax1.bar(df_compare['Annee'] - 0.2, df_compare['Permis_Urbanisme'], 
        width=0.4, color='coral', alpha=0.8, label='Permis urbanisme')
ax2.bar(df_compare['Annee'] + 0.2, df_compare['Permis_Environnement'], 
        width=0.4, color='steelblue', alpha=0.8, label="Permis environnement")
ax1.set_title("Comparaison : Permis d'urbanisme vs Permis d'environnement delivres\n(Ville de Bruxelles)", 
              fontweight='bold', fontsize=13)
ax1.set_xlabel("Annee")
ax1.set_ylabel("Permis d'urbanisme", color='coral')
ax2.set_ylabel("Permis d'environnement", color='steelblue')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
ax1.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

# Graph 2 : Ratio permis environnement / urbanisme
df_compare['ratio'] = df_compare.apply(
    lambda r: r['Permis_Environnement'] / r['Permis_Urbanisme'] if r['Permis_Urbanisme'] > 0 else 0, axis=1)
axes[1].plot(df_compare['Annee'], df_compare['ratio'], 
             linewidth=2.5, marker='o', color='green', markersize=6)
axes[1].axhline(df_compare['ratio'].mean(), color='gray', linestyle='--', 
                label=f"Moyenne: {df_compare['ratio'].mean():.2f}")
axes[1].fill_between(df_compare['Annee'], df_compare['ratio'], alpha=0.3, color='green')
axes[1].set_title("Ratio : permis d'environnement / permis d'urbanisme", 
                  fontweight='bold', fontsize=13)
axes[1].set_xlabel("Annee")
axes[1].set_ylabel("Ratio Env/Urba")
axes[1].legend()
axes[1].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig('03_comparaison_urba_env.png', dpi=150, bbox_inches='tight')
plt.show()
print("Graphique sauvegarde.")

In [ ]:
# Categories les plus frequentes en urbanisme et environnement
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Heatmap categories environnement par annee
cat_env_col = [c for c in df_env.columns if 'Cat' in c and 'gorie' in c and 'Code' not in c][0] if any('Cat' in c and 'gorie' in c and 'Code' not in c for c in df_env.columns) else None

if cat_env_col:
    df_env_cat = df_env.groupby([annee_col_env, cat_env_col])[total_col].sum().reset_index()
    top_env = df_env.groupby(cat_env_col)[total_col].sum().nlargest(8).index
    df_env_hm = df_env_cat[df_env_cat[cat_env_col].isin(top_env)].pivot_table(
        index=cat_env_col, columns=annee_col_env, values=total_col, aggfunc='sum').fillna(0)
    
    sns.heatmap(df_env_hm, ax=axes[0], cmap='Blues', annot=False, 
                linewidths=0.5, cbar_kws={'label': 'Nombre de permis'})
    axes[0].set_title("Heatmap : Categories de permis d'environnement\npar annee", fontweight='bold', fontsize=11)
    axes[0].set_xlabel("Annee")
    axes[0].set_ylabel("")
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].tick_params(axis='y', labelsize=8)
else:
    df_env.groupby(annee_col_env)[total_col].sum().plot(ax=axes[0], color='steelblue', linewidth=2)
    axes[0].set_title("Total permis environnement par annee", fontweight='bold')

# Top categories urbanisme
if cat_col_urba:
    top_urba_cat = df_urba.groupby(cat_col_urba)[total_urba].sum().nlargest(10).sort_values(ascending=True)
    axes[1].barh(top_urba_cat.index, top_urba_cat.values, 
                color=sns.color_palette("Oranges_r", len(top_urba_cat)))
    axes[1].set_title("Top 10 categories de permis d'urbanisme\n(cumul toutes annees)", fontweight='bold', fontsize=11)
    axes[1].set_xlabel("Total permis")
    axes[1].tick_params(axis='y', labelsize=8)
else:
    df_urba.groupby(domaine_urba)[total_urba].sum().sort_values(ascending=True).plot(kind='barh', ax=axes[1], color='coral')
    axes[1].set_title("Permis urbanisme par domaine", fontweight='bold')

plt.tight_layout()
plt.savefig('04_heatmap_categories.png', dpi=150, bbox_inches='tight')
plt.show()
print("Graphique sauvegarde.")

## 5. Interpretation politique et conclusions

### Constats principaux

1. **Dynamique de construction** : L'evolution des permis d'urbanisme reflete la pression immobiliere sur la ville. Les periodes de hausse correspondent souvent aux grands projets urbains (quartier europeen, Tour & Taxis, Canal).

2. **Regulation environnementale** : Le nombre de permis d'environnement delivres traduit l'activite des entreprises et commerces a impact environnemental. Une stabilisation ou baisse peut signaler une meilleure conformite ou une reduction des activites a risque.

3. **Ratio critique** : Lorsque le ratio permis urbanisme/environnement augmente fortement, cela peut indiquer que le developpement immobilier n'est pas accompagne d'une regulation environnementale proportionnelle.

4. **Resilience post-COVID** : La crise de 2020-2021 a clairement impacte les activites de construction et d'entreprise, visible dans la baisse des demandes de permis.

### Question politique repondue

> Y a-t-il une pression croissante sur l'environnement due au developpement immobilier a Bruxelles ?

**Reponse** : Les donnees montrent que la dynamique des permis d'urbanisme et environnement suit des tendances paralleles mais pas toujours proportionnelles. Des periodes de forte construction ne s'accompagnent pas systematiquement d'une augmentation des permis environnementaux, ce qui peut signaler soit une bonne internalisation des normes, soit un deficit de controle.

### Recommandations politiques

- Creer un tableau de bord croisant permis d'urbanisme et impact environnemental pour anticiper les risques.
- Renforcer l'evaluation environnementale systematique des grands projets urbains.
- Utiliser les donnees de permis comme indicateur avance pour guider les investissements en infrastructure verte.

---

**Contact pour decisions politiques ou formations IA & Data** :  
Kuate Joel Parfait — hello@dhcompany.pro | 0465 55 71 09  
[linkedin.com/in/joelparfaitkuate](https://be.linkedin.com/in/joelparfaitkuate/)  
www.axiatechnologie.com